# MLflow — Логирование экспериментов

Логируем оба эксперимента (Popularity Baseline и ALS) в MLflow для:
- Сравнения моделей
- Воспроизводимости экспериментов
- Версионирования артефактов

Запустить MLflow UI: `bash scripts/setup_mlflow.sh` → http://localhost:5000

In [ ]:
import os
import sys
from pathlib import Path

import mlflow

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODELS_DIR   = Path(os.getenv("MODELS_DIR", PROJECT_ROOT / "models"))
MLFLOW_DIR   = PROJECT_ROOT / "mlruns"

# На Windows str(Path) даёт C:\..., что MLflow читает как схему "c:" — ошибка.
# as_uri() возвращает file:///C:/... — правильный формат.
mlflow.set_tracking_uri(MLFLOW_DIR.as_uri())
mlflow.set_experiment("ecommerce-recsys")

print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Директория MLflow:   {MLFLOW_DIR}")
print(f"Эксперимент:         ecommerce-recsys")

## 1. Popularity Baseline

In [ ]:
with mlflow.start_run(run_name="popularity_baseline") as run:
    # Параметры
    mlflow.log_params({
        "model_type": "popularity",
        "top_k": 10,
        "target": "addtocart",
        "split": "time_based_14d_holdout",
        "n_popular_fallback": 200,
    })

    # Метрики (из ноутбука 02_models.ipynb)
    mlflow.log_metrics({
        "recall@10": 0.034,
        "ndcg@10": 0.021,
        "precision@10": 0.009,
    })

    # Артефакт
    artifact = MODELS_DIR / "artifact_popularity.pkl"
    if artifact.exists():
        mlflow.log_artifact(str(artifact), artifact_path="model")

    run_id_pop = run.info.run_id

print(f"Run ID (popularity): {run_id_pop}")
print("recall@10=0.034 | NDCG@10=0.021")

## 2. ALS v1

In [ ]:
with mlflow.start_run(run_name="als_v1") as run:
    # Параметры
    mlflow.log_params({
        "model_type": "als",
        "factors": 64,
        "iterations": 20,
        "regularization": 0.05,
        "random_state": 42,
        "top_k": 10,
        "target": "addtocart",
        "weights": "view=1, addtocart=5, transaction=10",
        "split": "time_based_14d_holdout",
        "n_popular_fallback": 200,
        "library": "implicit==0.7.2",
    })

    # Метрики (из ноутбука 02_models.ipynb)
    mlflow.log_metrics({
        "recall@10": 0.087,
        "ndcg@10": 0.056,
        "precision@10": 0.024,
    })

    # Артефакты
    for fname in ("als_model.pkl", "artifact.pkl"):
        path = MODELS_DIR / fname
        if path.exists():
            mlflow.log_artifact(str(path), artifact_path="model")

    run_id_als = run.info.run_id

print(f"Run ID (ALS v1): {run_id_als}")
print("recall@10=0.087 | NDCG@10=0.056")

## 3. Сравнение экспериментов

In [ ]:
import mlflow
import pandas as pd

client = mlflow.tracking.MlflowClient()
exp = mlflow.get_experiment_by_name("ecommerce-recsys")
runs = mlflow.search_runs(experiment_ids=[exp.experiment_id])

cols = ["tags.mlflow.runName", "metrics.recall@10", "metrics.ndcg@10", "metrics.precision@10"]
print("Эксперименты:")
print(runs[[c for c in cols if c in runs.columns]].to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

metrics_data = {
    "Модель": ["Popularity Baseline", "ALS v1"],
    "recall@10": [0.034, 0.087],
    "ndcg@10": [0.021, 0.056],
    "precision@10": [0.009, 0.024],
}

df = pd.DataFrame(metrics_data).set_index("Модель")
print(df)
print(f"\nУлучшение ALS vs Popularity:")
for col in ["recall@10", "ndcg@10", "precision@10"]:
    delta = (df.loc["ALS v1", col] / df.loc["Popularity Baseline", col] - 1) * 100
    print(f"  {col}: +{delta:.0f}%")

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, metric in zip(axes, ["recall@10", "ndcg@10", "precision@10"]):
    bars = ax.bar(df.index, df[metric], color=["steelblue", "seagreen"])
    ax.bar_label(bars, fmt="{:.4f}", padding=3)
    ax.set_title(metric)
    ax.set_ylim(0, df[metric].max() * 1.3)
    ax.tick_params(axis="x", rotation=15)

plt.suptitle("Сравнение моделей", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

## 4. Инструкции по воспроизведению

```bash
# Запустить MLflow UI
bash scripts/setup_mlflow.sh
# или
python -m mlflow ui --backend-store-uri ./mlruns

# Повторно залогировать эксперименты (после переобучения)
python scripts/log_mlflow_runs.py
```

**Лучшая модель: ALS v1**

| Метрика | Popularity | ALS v1 | Δ |
|---------|-----------|--------|---|
| recall@10 | 0.034 | **0.087** | +156% |
| ndcg@10 | 0.021 | **0.056** | +167% |
| precision@10 | 0.009 | **0.024** | +167% |

ALS значительно превосходит popularity baseline за счёт персонализации
(коллаборативная фильтрация учитывает паттерны совместных просмотров).
Cold-start покрыт popularity fallback.